In [27]:
from dotenv import load_dotenv

load_dotenv()

True

In [30]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown

# 1. Load Data
file = 'petshop.csv'
loader = CSVLoader(file_path=file)
docs = loader.load()

# 2. Create Vector Store & Retriever
# This replaces VectorstoreIndexCreator and DocArray
vectorstore = InMemoryVectorStore.from_documents(
    docs, 
    embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

# 3. Define the Prompt (Ensures high-quality Markdown output)
template = """Answer the user's question using only the provided context. 
If the information isn't in the context, say you don't know.

Context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# 4. Initialize the Chat Model
llm = ChatOpenAI(model="gpt-5.2", temperature=0)

# 5. Build the LCEL Chain
# This replaces index.query()
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 6. Execute
query = "Please list ALL animals with QUANTITY of more than 10."
response = rag_chain.invoke(query)

display(Markdown(response))

Goldfish (QUANTITY: 24)

In [31]:
docs

[Document(metadata={'source': 'petshop.csv', 'row': 0}, page_content='ID: 1\nANIMAL: Cat\nSALEPRICE: 450.09\nSALEDATE: 2018-05-29\nQUANTITY: 9'),
 Document(metadata={'source': 'petshop.csv', 'row': 1}, page_content='ID: 2\nANIMAL: Dog\nSALEPRICE: 666.66\nSALEDATE: 2018-06-01\nQUANTITY: 3'),
 Document(metadata={'source': 'petshop.csv', 'row': 2}, page_content='ID: 3\nANIMAL: Parrot\nSALEPRICE: 50.00\nSALEDATE: 2018-06-04\nQUANTITY: 2'),
 Document(metadata={'source': 'petshop.csv', 'row': 3}, page_content='ID: 4\nANIMAL: Hamster\nSALEPRICE: 60.60\nSALEDATE: 2018-06-11\nQUANTITY: 6'),
 Document(metadata={'source': 'petshop.csv', 'row': 4}, page_content='ID: 5\nANIMAL: Goldfish\nSALEPRICE: 48.48\nSALEDATE: 2018-06-14\nQUANTITY: 24')]

In [32]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [33]:
embed = embeddings.embed_query("Hi my name is Harrison")

In [34]:
print(len(embed))

1536


In [35]:
print(embed[:5])

[-0.021964654326438904, 0.006758837960660458, -0.01824948936700821, -0.03923514857888222, -0.014007173478603363]


In [47]:
db = InMemoryVectorStore.from_documents(
    docs, 
    embeddings
)

# 3. Perform Similarity Search
query = "Please suggest an ANIMAL with SALEPRICE of less than 55 dollars"

# This remains the same method name, but is more optimized under the hood
matched_docs = db.similarity_search(query, k=2)

# 4. Inspect the results
for doc in matched_docs:
    print(f"Content: {doc.page_content[:100]}...")

Content: ID: 1
ANIMAL: Cat
SALEPRICE: 450.09
SALEDATE: 2018-05-29
QUANTITY: 9...
Content: ID: 3
ANIMAL: Parrot
SALEPRICE: 50.00
SALEDATE: 2018-06-04
QUANTITY: 2...


In [48]:
matched_docs

[Document(id='01fe953f-359d-4e38-aefc-f0690fd2c4fc', metadata={'source': 'petshop.csv', 'row': 0}, page_content='ID: 1\nANIMAL: Cat\nSALEPRICE: 450.09\nSALEDATE: 2018-05-29\nQUANTITY: 9'),
 Document(id='97637dab-3e3e-4e79-95b7-08699f6e4ffe', metadata={'source': 'petshop.csv', 'row': 2}, page_content='ID: 3\nANIMAL: Parrot\nSALEPRICE: 50.00\nSALEDATE: 2018-06-04\nQUANTITY: 2')]

In [49]:
retriever = db.as_retriever()

In [51]:
llm = ChatOpenAI(temperature = 0.0, model="gpt-5.2")

In [52]:
qdocs = "".join([docs[i].page_content for i in range(len(docs))])

In [54]:
response = llm.invoke(f"{qdocs} Question: Please list all animals \
with QUANTITY of less than 5") 

In [57]:
display(Markdown(response.content))

Animals with **QUANTITY < 5**:

- **Dog** (QUANTITY: 3)  
- **Parrot** (QUANTITY: 2)

In [61]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Initialize the Chat Model
llm = ChatOpenAI(model="gpt-5.2", temperature=0)

# 5. Build the LCEL Chain
# This replaces index.query()
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 6. Execute
query = "Please list ALL animals with QUANTITY of more than 5."
response = rag_chain.invoke(query)

display(Markdown(response))

- Cat (QUANTITY: 9)  
- Goldfish (QUANTITY: 24)

In [62]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Load and Split (The 'Creator' used to do this internally)
docs = loader.load()

# 2. Setup Vector Store (Replaces VectorstoreIndexCreator + DocArray)
vectorstore = InMemoryVectorStore.from_documents(
    docs, 
    embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

# 3. Define the Chain logic (Replaces index.query)
prompt = ChatPromptTemplate.from_template("Answer based on context:\n{context}\n\n{question}")
llm = ChatOpenAI(model="gpt-4o", temperature=0)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

response = rag_chain.invoke("What is the summary?")

In [63]:
response

'The document contains sales data from a pet shop, detailing transactions for different animals. The records include:\n\n1. Cats were sold on May 29, 2018, with a sale price of $450.09 each, and a total quantity of 9.\n2. Dogs were sold on June 1, 2018, with a sale price of $666.66 each, and a total quantity of 3.\n3. Parrots were sold on June 4, 2018, with a sale price of $50.00 each, and a total quantity of 2.\n4. Hamsters were sold on June 11, 2018, with a sale price of $60.60 each, and a total quantity of 6.'